In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df = pd.read_csv('churn-bigml-80.csv')
df.isnull().sum()

In [ ]:
# Features and target
X = df.drop('Churn', axis=1)
y = df['Churn'].astype(int)                    # True → 1, False → 0


X = X.drop('State', axis=1)

# Define columns to encode
cat_cols = ['International plan', 'Voice mail plan']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)],
    remainder='passthrough'                    # now only numerical columns remain
)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),            # encode Yes/No columns
    ('scaler',       StandardScaler()),        # scale all remaining numerical columns
    ('classifier',   LogisticRegression(max_iter=1000))
])


model_pipeline.fit(X_train, y_train)



y_pred = model_pipeline.predict(X_test)
y_pred_prob = model_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print("\n=== MODEL PERFORMANCE ===")
print(f"Accuracy : {acc:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefficients = model_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print("\n=== Top 10 Features that INCREASE churn probability ===")
print(coef_df.head(10).round(4))
print("\n=== Top 10 Features that DECREASE churn probability ===")
print(coef_df.tail(10).round(4))